# Session 16: Capstone Review and Python Readiness

**Notebook 1 of 2: Concepts** · for revision before and after the last session (about 40 minutes)

The last session is mostly doing: a 60-minute practical assessment, and five-minute demonstrations of your capstones. This notebook covers the engineering the session asks of you, and that the capstone's last milestone builds:

1. **Distinguish** what unit, acceptance and integration tests each catch, and **write** an integration test that would have caught a real bug.
2. **Debug** under time pressure with a time-boxed **Observe → Hypothesize → Inspect → Fix → Verify** loop.
3. **Emit** structured logs with fields, and **correlate** the lines one request produces.
4. **Answer** operational questions from logs: latency percentiles, not-found rate, errors, cost.
5. **Present** an architecture and **defend** its decisions and limitations.

> **Kernel:** the course environment, after `uv sync`. **Cost: nothing.** Everything runs offline, on small examples built in the notebook. The examples are deliberately *not* the capstone's code, so the workbook is still yours to write.

## The engineering problem

Your capstone passes 136 acceptance tests. Is it ready for 300 engineers at 3 a.m.?

Not necessarily. The tests check each module against its specification, often with fakes. They don't check that the modules agree with **each other** when wired together, that a re-indexed document really changes the next answer, or that you could tell, from the outside, what the service did during an incident last night. And at the end of a course, the remaining question is whether **you** can do this without the scaffolding.

This session closes those gaps: tests across the whole system, logs that answer questions, and an assessment of what you can do on your own.

## Part 1: What integration tests catch

Here are two components, each with a passing test. One writes search results; the other formats them for the answer page.

In [ ]:
def search(question: str) -> list[dict]:
    """Returns hits as {"file": ..., "score": ...}, best first."""
    return [{"file": "runbooks/database-failover.md", "score": 0.81}]


def format_sources(hits: list[dict]) -> str:
    """Formats hits as '[1] runbooks/x.md (0.81)' lines."""
    return "\n".join(f"[{n}] {hit['source']} ({hit['score']:.2f})" for n, hit in enumerate(hits, 1))


# Each unit test uses data shaped the way *its* author assumed
assert search("failover")[0]["score"] == 0.81
assert format_sources([{"source": "a.md", "score": 0.5}]) == "[1] a.md (0.50)"
print("both unit tests pass")

try:
    print(format_sources(search("How do I fail over?")))
except KeyError as error:
    print("wired together:", type(error).__name__, error)

**Explain:** both tests pass, and the system is broken. One author called the field `file`, the other `source`. Each unit test fed its function data in the shape that function expected, so the disagreement only shows when the real output of one becomes the real input of the other.

That's the job of an **integration test**: wire real components together and check the behaviour a user sees. In the capstone you avoid this particular bug because both sides share one model (`Hit`, `Chunk`, `Answer`), which is one of the strongest arguments for typed models at module boundaries. Integration tests catch the rest: wiring, configuration, and behaviour that only exists across time, such as "after I edit a runbook and re-ingest, the next answer uses the new text".

| Level | Scope | Fakes | Speed | Catches | Misses |
|---|---|---|---|---|---|
| Unit | One function or class | Many | Milliseconds | Logic errors, edge cases | Disagreements between parts |
| Acceptance | One module, against its specification | Some (a scripted generator) | Fast | A module that doesn't do what was asked | Wiring and configuration |
| Integration | Several real modules wired together | Only external services (model, LLM) | Seconds | Contract mismatches, configuration, behaviour over time | Real-provider behaviour |
| End to end, live | The deployed system, the real LLM | None | Slow, costs money | What users will see | Rarely run; flaky |

**Engineering Insight:** keep the pyramid shape: many unit tests, fewer integration tests, very few live end-to-end checks. The capstone's integration tests still use the hashing embedder and the extractive generator, so they're deterministic, free and fast, and they still catch everything except the model's own behaviour, which is what the evaluation (Session 15) is for.

## Part 2: Debugging under time pressure

In the assessment, and during real incidents, you don't have time to read everything. The course's loop still works; what changes is that you **time-box** each step and write down what you've ruled out.

Here's a small pipeline with a bug that a user reported: *"I fixed the failover runbook an hour ago, re-indexed, and the assistant still gives the old answer."*

In [ ]:
from functools import lru_cache

index = {"failover": "Promote db-03 with promote-replica."}  # the "vector store"


def reindex(new_text: str) -> None:
    index["failover"] = new_text


@lru_cache(maxsize=256)
def search(question: str) -> str:
    """Somebody added a cache to make repeated questions faster."""
    return index["failover"] if "fail" in question.lower() else ""


print("before:", search("How do I fail over?"))
reindex("Promote db-07 with promote-replica.")
print("after re-index:", search("How do I fail over?"))

Work the loop, with a time box for each step:

1. **Observe (1 min):** the stale answer comes back for the *same* question after re-indexing. Does a differently-worded question get the new text? (Try it in a new cell.)
2. **Hypothesize (1 min), and write them down:** the re-index didn't happen; the re-index wrote somewhere else; something between the index and the answer remembers old results.
3. **Inspect (3 min):** check the cheapest hypothesis first. Is the new text in `index`? What does `search.cache_info()` say?
4. **Fix:** the cache is keyed by the question only, so it can't know the index changed.
5. **Verify:** the same question returns the new text, and a test proves it.

In [ ]:
print("index holds:", index["failover"])
print(search.cache_info())


# One fix: clear the cache whenever the index changes
def reindex(new_text: str) -> None:
    index["failover"] = new_text
    search.cache_clear()


reindex("Promote db-07 with promote-replica.")
assert search("How do I fail over?") == "Promote db-07 with promote-replica."
print("fixed:", search("How do I fail over?"))

**Explain:** `cache_info()` showing hits was the evidence; without it, you'd be reading the re-index code for an hour. The general lesson: **any cache must be invalidated by every change to what it caches**, and a test that changes the data and asks again is the only reliable check. That's exactly the shape of the capstone's integration test *"an edited runbook is answered from its new text"*.

**Common Mistake:** fixing the symptom and moving on. Before you change code, write the test that fails; after the fix, watch it pass. Under time pressure, that's still the fastest way to know you're done.

## Part 3: Structured logs

A log line written for a human, such as `INFO answered from 2 of 4 sources (gpt-5.6-terra)`, is easy to read and hard to query. How many answers were "not found" yesterday? What was the slowest request? With text, you'd write a regular expression per message, and it would break the next time someone reworded one.

**Structured logging** writes the *fields*, not a sentence: each line has an event name and named values, usually as JSON, so a log system can filter, count and graph them. Python's `logging` already carries the data you need; you choose how a record is **formatted**.

The capstone's workbook asks for JSON. Here's the same idea in another common format, *logfmt* (`key=value` pairs), so you can see the mechanics without the workbook's answer:

In [ ]:
import io
import logging

STANDARD = set(vars(logging.LogRecord("", 0, "", 0, "", None, None))) | {"message"}


class LogfmtFormatter(logging.Formatter):
    def format(self, record: logging.LogRecord) -> str:
        fields = {"level": record.levelname, "logger": record.name, "msg": record.getMessage()}
        fields |= {k: v for k, v in vars(record).items() if k not in STANDARD}
        return " ".join(
            f"{key}={value!r}" if isinstance(value, str) else f"{key}={value}"
            for key, value in fields.items()
        )


stream = io.StringIO()
handler = logging.StreamHandler(stream)
handler.setFormatter(LogfmtFormatter())
demo = logging.getLogger("demo.answering")
demo.addHandler(handler)
demo.setLevel(logging.INFO)
demo.propagate = False

demo.info("answer", extra={"event": "answer", "found": True, "sources": 2, "duration_ms": 412.5})
print(stream.getvalue())

- **`extra={...}`** attaches fields to the record as attributes. The formatter finds them by comparing the record's attributes with those every `LogRecord` has.
- **The message stays short and constant** (`"answer"`), and the facts go in fields. That's what makes lines countable.
- **What you don't log matters as much.** No question, no answer text, no document content: the same rule as Session 14, now enforced by design, because there's no field for them.

### One request, many lines

A single `POST /ask` produces several log lines from different modules: the request, the retrieval, the answer. To find them together, every line needs the same **request ID**. Passing an ID through every function signature would be intrusive; Python's **`contextvars`** module gives each request its own value of a variable, even when many requests run concurrently, which is how the capstone's provided `REQUEST_ID` works.

In [ ]:
import asyncio
import contextvars

request_id = contextvars.ContextVar("request_id", default=None)


async def answer(question: str) -> str:
    await asyncio.sleep(0.01)  # other requests run here
    return f"[{request_id.get()}] answered {len(question)}-character question"


async def handle(rid: str, question: str) -> str:
    request_id.set(rid)
    return await answer(question)


async def main() -> None:
    print(
        *await asyncio.gather(
            handle("req-A", "How do I fail over?"), handle("req-B", "Who approves changes?")
        ),
        sep="\n",
    )


await main()

Two requests ran interleaved, and each still saw its own ID: `asyncio` tasks (and the threads FastAPI runs sync endpoints in) each get a copy of the context. In Java or .NET, the same idea is the logging framework's *Mapped Diagnostic Context* (SLF4J's MDC) or `ILogger.BeginScope`; in Python, `contextvars` is the building block, and a log-record factory or filter copies the value into every record.

## Part 4: Questions your logs can answer

Once the lines are structured, operational questions become a few lines of Python (or a query in your log system). Here's a synthetic day of `answer` events, generated deterministically, and the questions an on-call engineer would ask about the assistant itself.

In [ ]:
import json
import random
import statistics

random.seed(16)
lines = []
for i in range(400):
    found = random.random() > 0.12
    lines.append(
        json.dumps(
            {
                "event": "answer",
                "found": found,
                "model": "gpt-5.6-terra" if found or random.random() > 0.5 else None,
                "input_tokens": random.randint(900, 1500) if found else 0,
                "output_tokens": random.randint(40, 160) if found else 0,
                "duration_ms": round(random.lognormvariate(6.6, 0.45), 1),
            }
        )
    )

events = [json.loads(line) for line in lines]
durations = [e["duration_ms"] for e in events]
p50, p95 = (
    statistics.quantiles(durations, n=100, method="inclusive")[49],
    statistics.quantiles(durations, n=100, method="inclusive")[94],
)
not_found = sum(not e["found"] for e in events) / len(events)
tokens_in = sum(e["input_tokens"] for e in events)
tokens_out = sum(e["output_tokens"] for e in events)

print(f"answers: {len(events)}")
print(f"not found: {not_found:.1%}")
print(f"latency p50 {p50:.0f} ms, p95 {p95:.0f} ms, max {max(durations):.0f} ms")
print(f"tokens: {tokens_in:,} in, {tokens_out:,} out")

**Predict, then check:** if the not-found rate jumped from 12% to 40% the morning after a deployment, what would you look at first? (Answer in the quiz.)

| Question | Field | Typical alert |
|---|---|---|
| Is it slow? | `duration_ms`, by percentile | p95 above an agreed limit for 10 minutes |
| Is it answering? | `found` | not-found rate far from its usual level, **in either direction** |
| Is it failing? | the request's `status` | 5xx rate above 1% |
| What does it cost? | `input_tokens`, `output_tokens` | daily tokens above budget; multiply by your model's price per million |
| What happened to *this* user? | `request_id` | (not an alert: a search) |

**Engineering Insight:** use percentiles, not averages, for latency. Averages hide the slow tail that users complain about. And note `method="inclusive"` in the cell: Session 13 showed that `statistics.quantiles`' default can extrapolate beyond your data on small samples.

## Part 5: Presenting and defending an architecture

You'll present your capstone in five minutes. Experienced reviewers listen for three things: whether you understand the problem, whether your decisions were **decisions** (with alternatives and reasons), and whether you know where the system is weak.

A simple structure that works:

1. **The problem, in one sentence,** and who has it.
2. **The architecture,** on one diagram of *your* code: modules and the path of one question.
3. **Two or three decisions,** each with the alternative you rejected and why.
4. **Evidence:** a live answer, a live "not found", and one number from your evaluation.
5. **Limitations,** and what you'd do next, in priority order.

| Weak | Strong |
|---|---|
| "I used Chroma because it's popular." | "I needed a persistent local store with metadata filters. Chroma does both; the cost is that its `add` silently ignores existing IDs, which I handle with replace." |
| "It handles errors." | "An LLM outage returns 503 and the UI says try again; a rejected key returns 500 and stops the service at startup. Tests prove both." |
| "The evaluation is 94%." | "Retrieval finds the right file for 15 of 16 answerable questions. The miss is a vocabulary gap: the policy says 'response targets', the question says 'resolve'." |
| "There are no known issues." | "The threshold was chosen on 20 questions; it needs more. Re-ingestion isn't safe if two runs overlap. Logs aren't shipped anywhere yet." |

**Common Mistake:** spending four of the five minutes on a feature tour. Show less, and explain more.

## Part 6: Readiness, and what comes next

The final assessment asks you to do, alone and in an hour, a small version of what the course taught: a `uv` project, data handling, a Pydantic contract behind FastAPI, a structured LLM call with honest error handling, a debugging fix with tests, and short written explanations. [`assessments/python-readiness.md`](../../../assessments/python-readiness.md) describes the standard and how to prepare.

What this course deliberately didn't cover, and where your Python now lets you go: evaluation frameworks for LLM systems, agents and tool use, advanced retrieval (hybrid search, re-ranking), fine-tuning, and production deployment and monitoring. Each of those starts from exactly the skills you've just used: typed contracts, tested integrations, honest failures and measurement.

## Key takeaways

- Unit and acceptance tests check parts; **integration tests** check that the parts agree, and behaviour over time.
- Under pressure, **time-box** Observe → Hypothesize → Inspect → Fix → Verify, and write the failing test before the fix.
- **Structured logs** carry fields, not sentences; a **request ID** in a context variable ties a request's lines together.
- Logs answer operational questions: **percentiles** for latency, rates for answers and errors, tokens for cost.
- A good presentation explains **decisions, evidence and limitations**, not features.

## Further reading

- [Logging HOWTO](https://docs.python.org/3.13/howto/logging.html) and the [Logging Cookbook](https://docs.python.org/3.13/howto/logging-cookbook.html), including adding contextual information to records.
- [`contextvars`](https://docs.python.org/3.13/library/contextvars.html) in the Python 3.13 documentation.
- The session's [references](../resources/references.md) list more.